### 1. Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

formatted_prompt = prompt.invoke({
    "topic": "LangChain"
})
formatted_prompt2 = prompt.invoke({
    "topic": "Python"
})

print(formatted_prompt)
print(formatted_prompt2)

### 2. PromptTemplate + Gemini

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

formatted_prompt = prompt.invoke({
    "topic": "LangChain"
})

response = llm.invoke(formatted_prompt)

print(response.text)

### 3. Multiple variables

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = PromptTemplate.from_template(
    """
    Explain {topic} for a {level} student.
    Give {count} examples.
    """
)

formatted_prompt = prompt.invoke({
    "topic": "RAG",
    "level": "beginner",
    "count": 3
})

response = llm.invoke(formatted_prompt)

print(formatted_prompt.text)
print(response.text)

### 4. Chat Prompt Template

- System → instructions
- Human  → user,
- AI → previous AI response

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful {role}."
    ),
    (
        "human",
        "Explain {topic}."
    )
])

In [ ]:
messages = prompt.invoke({
    "role": "Python teacher",
    "topic": "decorators"
})

print(messages)

In [ ]:
chain = prompt | llm    # This is LCLE explaied below

response = chain.invoke({
    "role": "teacher",
    "topic": "recursion"
})

print(response.text)

**Prompt | LLM**

This is **LCEL (LangChain Expression Language)**.

**Why `prompt | llm`?**

Think of it like a pipeline:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Response
```


--- 

In code:
```python
chain = prompt | llm
``` 
---

Then:
```python
chain.invoke({
  "topic": "RAG"
})
```

- This is the modern LangChain style.
- You will see this pattern everywhere in LangChain applications.

### 5. System + Human + variable

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an expert software engineer.
        Explain concepts simply.
        Give practical examples.
        """
    ),
    (
        "human",
        "{question}"
    )
])

chain = prompt | llm

response = chain.invoke({
    "question": "What is Redis?"
})

print(response.text)

### 6. Few-shot prompting
(a model example)
- Input: happy
- Output: positive
---
- Input: terrible
- Output: negative
---
- Input: amazing
- Output: ?

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Classify the sentiment as positive or negative."
    ),

    (
        "human",
        "I love this product."
    ),
    (
        "ai",
        "positive"
    ),

    (
        "human",
        "This product is terrible."
    ),
    (
        "ai",
        "negative"
    ),

    (
        "human",
        "{text}"
    )
])

chain = prompt | llm

response = chain.invoke({
    "text": "This is an amazing product!"
})

print(response.text)

### 7. Dynamic few-shot examples

In [ ]:
# Later, LangChain has dedicated few-shot prompt classes.
# But don't start there.

# First understand:  few-shot = examples inside prompt
# Then we'll learn:   FewShotChatMessagePromptTemplate

# when you actually need dynamic example selection.
# That's more useful for advanced applications than memorizing the older FewShotPromptTemplate API.

### 8. Conversation history

In [ ]:
# Now suppose:

# Human: My name is Alok.
# AI: Nice to meet you!

# Human: What is my name?

# The second question requires history.

# A chat prompt can include a placeholder for previous messages.

In [ ]:
# A chat prompt can include a placeholder for previous messages.

from langchain_core.prompts import ( ChatPromptTemplate, MessagesPlaceholder )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),

    MessagesPlaceholder( variable_name="history" ),

    (
        "human",
        "{question}"
    )
])

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content="My name is Alok."),
    AIMessage(content="Nice to meet you, Alok!")
]

messages = prompt.invoke({
    "history": history,
    "question": "What is my name?"
})

response = llm.invoke(messages)

print(response.text)

### 9. Prompt + streaming

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful teacher."
    ),
    (
        "human",
        "Explain {topic} in simple and short."
    )
])

chain = prompt | llm

for chunk in chain.stream({ "topic": "RAG" }):
    print(chunk.text, end="", flush=True)

### 10. Output parsing

In [ ]:
# Now an important problem.

# Gemini returns:

# "Python is a programming language..."

# But your application might need:

# {
#     "answer": "...",
#     "difficulty": "easy"
# }

# That's where output parsing / structured output comes in.

In [26]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

class TopicInfo(BaseModel):
    summary: str = Field(description="Short explanation")
    difficulty: str = Field(description="easy, medium, or hard")
    key_points: list[str]

structured_llm = llm.with_structured_output(TopicInfo)

response = structured_llm.invoke(
    "Explain RAG"
)

print(response)

summary='Retrieval-Augmented Generation (RAG) is a technique that improves the accuracy and reliability of generative AI models by fetching facts from an external knowledge base before generating a response.' difficulty='medium' key_points=['Combines information retrieval with text generation', 'Reduces hallucinations in large language models', 'Allows models to access up-to-date or private data without retraining']


### 11. Prompt + structured output

In [30]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an AI tutor."
    ),
    (
        "human",
        "Explain {topic}."
    )
])

class TopicInfo(BaseModel):
    summary: str = Field(description="One line short explanation")
    difficulty: str = Field(description="easy, medium, or hard")
    key_points: list[str]

structured_llm = llm.with_structured_output(
    TopicInfo
)

chain = prompt | structured_llm

response = chain.invoke({
    "topic": "FIFO"
})

print(response)

summary='First-In, First-Out (FIFO) is a method for organizing data processing or inventory management where the oldest entry is processed first.' difficulty='easy' key_points=['Operates on the principle that the first element in is the first element out.', 'Commonly used in data structures like queues.', 'Widely applied in inventory valuation and supply chain management to prevent stock from spoiling.']
